# Tennis domain adaptation experiments

This Colab notebook is an orchestration layer for the repository scripts. It does not duplicate model loading, generation, scoring, or training logic. Use a GPU runtime for any model evaluation, and start with smoke-mode commands before enabling full runs.

## 1. Runtime setup

Recommended Colab runtime: GPU. The cells below check GPU visibility, mount Google Drive, set the project path, optionally clone the repository, and move into the repo root.

In [1]:
from shutil import which
import subprocess

if which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found; continuing without a visible NVIDIA GPU.")

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Google Drive mount skipped: {exc}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
import subprocess

# Edit this path for your Drive layout or cloned checkout.
PROJECT_ROOT = Path('/content/drive/Othercomputers/My Mac/Desktop/Folders/Documents n Stuff/Polito/DNLP/Project/tiser_temporal_reasoning_extension')

# Optional. Set to your GitHub repo URL if PROJECT_ROOT does not already exist.
GITHUB_REPO_URL = ''

if not PROJECT_ROOT.exists():
    if GITHUB_REPO_URL:
        PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(f'git clone "{GITHUB_REPO_URL}" "{PROJECT_ROOT}"', shell=True, check=True)
    else:
        raise FileNotFoundError(
            f'{PROJECT_ROOT} does not exist. Set PROJECT_ROOT to your repo path or set GITHUB_REPO_URL.'
        )

print(PROJECT_ROOT)

/content/drive/Othercomputers/My Mac/Desktop/Folders/Documents n Stuff/Polito/DNLP/Project/tiser_temporal_reasoning_extension


In [4]:
%cd {PROJECT_ROOT}

/content/drive/Othercomputers/My Mac/Desktop/Folders/Documents n Stuff/Polito/DNLP/Project/tiser_temporal_reasoning_extension


## 2. Install dependencies

In [5]:
import os
import shlex
import subprocess
import sys
import textwrap

def normalize_command(command):
    command = textwrap.dedent(command).strip()
    command = command.replace(" \\\n", " ")
    return " ".join(line.strip() for line in command.splitlines() if line.strip())

def run_shell(command, *, enabled=True):
    command = normalize_command(command)
    if not enabled:
        print("[skip] Stage disabled. Command retained for reproducibility:")
        print(command)
        return
    print("[run]")
    print(command)
    args = shlex.split(command, posix=True)
    if args and args[0] == "python":
        args[0] = sys.executable
    subprocess.run(args, check=True)

In [6]:
import importlib.util
import subprocess
import sys

runtime_dependencies = {
    'accelerate': 'accelerate',
    'bitsandbytes': 'bitsandbytes',
    'datasets': 'datasets',
    'peft': 'peft',
    'torch': 'torch',
    'transformers': 'transformers',
    'trl': 'trl',
    'yaml': 'pyyaml',
}
missing = [package for module, package in runtime_dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing runtime dependencies:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
else:
    print('All checked runtime dependencies are importable.')

All checked runtime dependencies are importable.


In [7]:
import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))

Python: 3.12.13
Torch: 2.11.0+cu128
CUDA available: True
CUDA device: NVIDIA A100-SXM4-40GB


## 3. Configuration flags

Full 7B evaluations and training are disabled by default. Enable only the stages you intend to run.

In [8]:
RUN_SMOKE = False
RUN_BASE_QWEN = False
RUN_ORIGINAL_TISER = False
RUN_TENNIS_ONLY_EVAL = False
RUN_MIXED_REPLAY_EVAL = False
RUN_AGGREGATION = False
RUN_TRAINING = True
RUN_EXPERIMENT_PLAN = False

LIMIT = 5
BATCH_SIZE = 1
MAX_NEW_TOKENS = 256

## 4. Path configuration

In [9]:
TENNIS_TEST = 'data/tennis/tennis_test.json'
TENNIS_TRAIN_TRACED = 'data/tennis/tennis_train_traced_full.json'

ORIGINAL_TISER_ADAPTER = 'checkpoints/adapter'
TENNIS_ONLY_ADAPTER = 'model/tennis_only_qwen7b/adapter'
MIXED_REPLAY_ADAPTER = 'model/mixed_tennis_tiser_replay_qwen7b/adapter'

CONFIG = 'config/config_tennis.yaml'
SMOKE_CONFIG = 'config/config_tennis_smoke.yaml'

RESULTS_DIR = 'results/tennis_domain_adaptation'

In [10]:
import subprocess
import textwrap

def run_shell(command, *, enabled=True):
    command = textwrap.dedent(command).strip()
    if not enabled:
        print('[skip] Stage disabled. Command retained for reproducibility:')
        print(command)
        return
    print('[run]')
    print(command)
    subprocess.run(command, shell=True, check=True)

## 5. Preflight checks

In [11]:
from pathlib import Path
from IPython.display import Markdown, display

def preflight_row(label, path, *, kind='file', required=True, enabled=True):
    p = Path(path)
    exists = p.is_dir() if kind == 'dir' else p.exists()
    if exists:
        status = 'PASS'
        note = 'found'
    elif enabled and required:
        status = 'FAIL'
        note = 'missing and required for enabled stage'
    else:
        status = 'WARN'
        note = 'missing; stage is disabled or script is optional'
    return {'status': status, 'item': label, 'path': str(p), 'note': note}

checks = [
    preflight_row('tennis test file', TENNIS_TEST),
    preflight_row('tennis train traced file', TENNIS_TRAIN_TRACED, required=True, enabled=RUN_TRAINING),
    preflight_row('config', CONFIG),
    preflight_row('smoke config', SMOKE_CONFIG),
    preflight_row('original TISER adapter', ORIGINAL_TISER_ADAPTER, kind='dir', required=True, enabled=RUN_TRAINING),
    preflight_row('tennis-only adapter', TENNIS_ONLY_ADAPTER, kind='dir', enabled=RUN_TENNIS_ONLY_EVAL),
    preflight_row('mixed replay adapter', MIXED_REPLAY_ADAPTER, kind='dir', enabled=RUN_MIXED_REPLAY_EVAL),
    preflight_row('evaluate_tennis.py', 'scripts/tennis/evaluate_tennis.py'),
    preflight_row('compare_adapters.py', 'scripts/tennis/compare_adapters.py'),
    preflight_row('run_experiment_plan.py', 'scripts/tennis/run_experiment_plan.py', required=False, enabled=False),
    preflight_row('aggregate_tennis_results.py', 'scripts/tennis/aggregate_tennis_results.py', required=False, enabled=False),
    preflight_row('train_tennis.py', 'scripts/tennis/train_tennis.py', required=True, enabled=RUN_TRAINING),
]

header = '| Status | Item | Path | Note |\n|---|---|---|---|'
body = '\n'.join(f"| {row['status']} | {row['item']} | `{row['path']}` | {row['note']} |" for row in checks)
display(Markdown(header + '\n' + body))

failures = [row for row in checks if row['status'] == 'FAIL']
if failures:
    raise FileNotFoundError('Preflight failed for: ' + ', '.join(row['item'] for row in failures))

| Status | Item | Path | Note |
|---|---|---|---|
| PASS | tennis test file | `data/tennis/tennis_test.json` | found |
| PASS | tennis train traced file | `data/tennis/tennis_train_traced_full.json` | found |
| PASS | config | `config/config_tennis.yaml` | found |
| PASS | smoke config | `config/config_tennis_smoke.yaml` | found |
| PASS | original TISER adapter | `checkpoints/adapter` | found |
| WARN | tennis-only adapter | `model/tennis_only_qwen7b/adapter` | missing; stage is disabled or script is optional |
| WARN | mixed replay adapter | `model/mixed_tennis_tiser_replay_qwen7b/adapter` | missing; stage is disabled or script is optional |
| PASS | evaluate_tennis.py | `scripts/tennis/evaluate_tennis.py` | found |
| PASS | compare_adapters.py | `scripts/tennis/compare_adapters.py` | found |
| PASS | run_experiment_plan.py | `scripts/tennis/run_experiment_plan.py` | found |
| PASS | aggregate_tennis_results.py | `scripts/tennis/aggregate_tennis_results.py` | found |
| PASS | train_tennis.py | `scripts/tennis/train_tennis.py` | found |

In [12]:
!pip install -r requirements.txt

## 6. Smoke evaluation

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --condition base_qwen_smoke \
  --no-adapter \
  --limit {LIMIT} \
  --batch-size 1 \
  --max-new-tokens 256 \
  --output-dir "{RESULTS_DIR}/scored/base_qwen_smoke"
''', enabled=RUN_SMOKE)

## 7. Full baseline evaluations

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --condition base_qwen \
  --no-adapter \
  --batch-size {BATCH_SIZE} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --output-dir "{RESULTS_DIR}/scored/base_qwen"
''', enabled=RUN_BASE_QWEN)

NameError: name 'run_shell' is not defined

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --adapter-dir "{ORIGINAL_TISER_ADAPTER}" \
  --condition original_tiser \
  --batch-size {BATCH_SIZE} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --output-dir "{RESULTS_DIR}/scored/original_tiser"
''', enabled=RUN_ORIGINAL_TISER)

## 8. Tennis adapter evaluations

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --adapter-dir "{TENNIS_ONLY_ADAPTER}" \
  --condition tennis_only \
  --batch-size {BATCH_SIZE} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --output-dir "{RESULTS_DIR}/scored/tennis_only"
''', enabled=RUN_TENNIS_ONLY_EVAL)

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --adapter-dir "{MIXED_REPLAY_ADAPTER}" \
  --condition mixed_tennis_tiser_replay \
  --batch-size {BATCH_SIZE} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --output-dir "{RESULTS_DIR}/scored/mixed_tennis_tiser_replay"
''', enabled=RUN_MIXED_REPLAY_EVAL)

## 9. Experiment plan runner

This writes `results/tennis_domain_adaptation/comparisons/run_tennis_experiments.sh` in dry-run mode when the plan script exists. It does not execute the generated full experiment script.

In [ ]:
if Path('scripts/tennis/run_experiment_plan.py').exists():
    run_shell(f'''
python scripts/tennis/run_experiment_plan.py \
  --config "{CONFIG}" \
  --tennis-test "{TENNIS_TEST}" \
  --original-tiser-adapter "{ORIGINAL_TISER_ADAPTER}" \
  --tennis-adapter "{TENNIS_ONLY_ADAPTER}" \
  --mixed-adapter "{MIXED_REPLAY_ADAPTER}" \
  --limit {LIMIT}
''', enabled=RUN_EXPERIMENT_PLAN)
else:
    print('scripts/tennis/run_experiment_plan.py not found; skipping dry-run plan generation.')

In [ ]:
from datetime import datetime

RUN_NAME = f"tennis_from_tiser_qwen7b_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
TENNIS_ONLY_ADAPTER = f"model/{RUN_NAME}/adapter"

!python scripts/tennis/train_tennis.py \
  --config "{CONFIG}" \
  --train-file "{TENNIS_TRAIN_TRACED}" \
  --base-adapter "{ORIGINAL_TISER_ADAPTER}" \
  --run-name "{RUN_NAME}"


[tennis-train] validated data/tennis/tennis_train_traced_full.json: 600 records, answer match 100.00%
[tennis-train] run_name: tennis_from_tiser_qwen7b_20260616_093822
[tennis-train] train_file: data/tennis/tennis_train_traced_full.json
[tennis-train] base_adapter: checkpoints/adapter
[tennis-train] output_run_dir: outputs/tennis_from_tiser_qwen7b_20260616_093822
[tennis-train] adapter_dir: model/tennis_from_tiser_qwen7b_20260616_093822/adapter
2026-06-16 09:38:27.020261: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-16 09:38:27.091100: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in

In [13]:
# GRID-SEARCH

from datetime import datetime
from pathlib import Path
import json
import subprocess
import sys
import yaml
from math import prod

GRID_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
GRID_ROOT = Path("gridsearch")
GRID_DIR = GRID_ROOT / f"tennis_from_tiser_qwen7b_{GRID_ID}"
CONFIG_DIR = GRID_DIR / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

base_cfg = yaml.safe_load(Path(CONFIG).read_text())

epochs_grid = [1, 2, 3]
learning_rate_grid = [5.0e-5, 1.0e-4, 2.0e-4]
batch_grid = [
    {"per_device_batch_size": 4, "gradient_accumulation_steps": 4},
    {"per_device_batch_size": 4, "gradient_accumulation_steps": 8},
]
lora_grid = [{"r": 16, "alpha": 32}]
dropout_grid = [0.05]

total_runs = prod([
    len(epochs_grid),
    len(learning_rate_grid),
    len(batch_grid),
    len(lora_grid),
    len(dropout_grid),
])

def combo_key(epochs, lr, batch_cfg, lora_cfg, dropout):
    return (
        f"e={epochs}|"
        f"lr={lr:g}|"
        f"bs={batch_cfg['per_device_batch_size']}|"
        f"ga={batch_cfg['gradient_accumulation_steps']}|"
        f"r={lora_cfg['r']}|"
        f"alpha={lora_cfg['alpha']}|"
        f"dropout={dropout:g}"
    )

def adapter_complete(adapter_dir):
    adapter_dir = Path(adapter_dir)
    return (
        adapter_dir.is_dir()
        and (adapter_dir / "adapter_config.json").exists()
        and (adapter_dir / "adapter_model.safetensors").exists()
    )

completed = {}

for manifest_path in GRID_ROOT.glob("tennis_from_tiser_qwen7b_*/manifest.json"):
    try:
        rows = json.loads(manifest_path.read_text())
    except Exception as exc:
        print(f"[warn] could not read {manifest_path}: {exc}")
        continue

    for row in rows:
        key = row.get("combo_key")
        if not key:
            key = (
                f"e={row.get('epochs')}|"
                f"lr={row.get('learning_rate'):g}|"
                f"bs={row.get('per_device_batch_size')}|"
                f"ga={row.get('gradient_accumulation_steps')}|"
                f"r={row.get('lora_r')}|"
                f"alpha={row.get('lora_alpha')}|"
                f"dropout={row.get('dropout'):g}"
            )

        if row.get("status") == "ok" and adapter_complete(row.get("adapter_dir", "")):
            completed[key] = row

# Also scan model/*/adapter/run_meta.json so completed runs are detected
# even if their gridsearch manifest is missing.
for run_meta_path in Path("model").glob("*/adapter/run_meta.json"):
    try:
        run_meta = json.loads(run_meta_path.read_text())
        cfg = run_meta["config"]
        train_cfg = cfg["train"]
        lora_cfg = cfg["lora"]

        key = (
            f"e={train_cfg.get('num_epochs')}|"
            f"lr={train_cfg.get('learning_rate'):g}|"
            f"bs={train_cfg.get('per_device_batch_size')}|"
            f"ga={train_cfg.get('gradient_accumulation_steps')}|"
            f"r={lora_cfg.get('r')}|"
            f"alpha={lora_cfg.get('alpha')}|"
            f"dropout={lora_cfg.get('dropout'):g}"
        )

        adapter_dir = run_meta_path.parent
        if adapter_complete(adapter_dir):
            completed[key] = {
                "combo_key": key,
                "run_name": adapter_dir.parent.name,
                "adapter_dir": adapter_dir.as_posix(),
                "epochs": train_cfg.get("num_epochs"),
                "learning_rate": train_cfg.get("learning_rate"),
                "per_device_batch_size": train_cfg.get("per_device_batch_size"),
                "gradient_accumulation_steps": train_cfg.get("gradient_accumulation_steps"),
                "lora_r": lora_cfg.get("r"),
                "lora_alpha": lora_cfg.get("alpha"),
                "dropout": lora_cfg.get("dropout"),
                "status": "ok",
                "source": "model_run_meta",
            }
    except Exception as exc:
        print(f"[warn] could not read {run_meta_path}: {exc}")

manifest = []
run_index = 0
skipped = 0

for epochs in epochs_grid:
    for lr in learning_rate_grid:
        for batch_cfg in batch_grid:
            for lora_cfg in lora_grid:
                for dropout in dropout_grid:
                    run_index += 1
                    key = combo_key(epochs, lr, batch_cfg, lora_cfg, dropout)

                    if key in completed:
                        skipped += 1
                        previous = completed[key]
                        print(
                            f"\n=== Grid run {run_index}/{total_runs}: SKIP existing combo ===\n"
                            f"{key}\n"
                            f"existing run: {previous.get('run_name')}\n"
                            f"adapter: {previous.get('adapter_dir')}"
                        )
                        manifest.append({
                            "run_index": run_index,
                            "combo_key": key,
                            "run_name": previous.get("run_name"),
                            "adapter_dir": previous.get("adapter_dir"),
                            "epochs": epochs,
                            "learning_rate": lr,
                            **batch_cfg,
                            **{f"lora_{k}": v for k, v in lora_cfg.items()},
                            "dropout": dropout,
                            "status": "skipped_existing",
                        })
                        (GRID_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
                        continue

                    run_name = (
                        f"tennis_from_tiser"
                        f"_e{epochs}"
                        f"_lr{lr:g}"
                        f"_bs{batch_cfg['per_device_batch_size']}"
                        f"_ga{batch_cfg['gradient_accumulation_steps']}"
                        f"_r{lora_cfg['r']}"
                        f"_a{lora_cfg['alpha']}"
                        f"_d{str(dropout).replace('.', 'p')}"
                        f"_{GRID_ID}_{run_index:03d}"
                    )

                    cfg = json.loads(json.dumps(base_cfg))
                    cfg["run_name"] = run_name
                    cfg["train"]["num_epochs"] = epochs
                    cfg["train"]["learning_rate"] = lr
                    cfg["train"]["per_device_batch_size"] = batch_cfg["per_device_batch_size"]
                    cfg["train"]["gradient_accumulation_steps"] = batch_cfg["gradient_accumulation_steps"]
                    cfg["lora"]["r"] = lora_cfg["r"]
                    cfg["lora"]["alpha"] = lora_cfg["alpha"]
                    cfg["lora"]["dropout"] = dropout

                    cfg_path = CONFIG_DIR / f"{run_name}.yaml"
                    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

                    adapter_dir = f"model/{run_name}/adapter"

                    command = [
                        sys.executable,
                        "scripts/tennis/train_tennis.py",
                        "--config", str(cfg_path),
                        "--train-file", TENNIS_TRAIN_TRACED,
                        "--base-adapter", ORIGINAL_TISER_ADAPTER,
                        "--run-name", run_name,
                    ]

                    manifest_row = {
                        "run_index": run_index,
                        "combo_key": key,
                        "run_name": run_name,
                        "config_path": str(cfg_path),
                        "adapter_dir": adapter_dir,
                        "epochs": epochs,
                        "learning_rate": lr,
                        **batch_cfg,
                        **{f"lora_{k}": v for k, v in lora_cfg.items()},
                        "dropout": dropout,
                        "status": "pending",
                    }
                    manifest.append(manifest_row)
                    (GRID_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

                    print(f"\n=== Grid run {run_index}/{total_runs}: {run_name} ===")
                    result = subprocess.run(command)

                    manifest_row["status"] = "ok" if result.returncode == 0 else "failed"
                    (GRID_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

                    if result.returncode != 0:
                        raise RuntimeError(f"Grid run failed: {run_name}")

print(f"Grid complete. Manifest: {GRID_DIR / 'manifest.json'}")
print(f"Skipped existing combinations: {skipped}")


=== Grid run 1/18: tennis_from_tiser_e1_lr5e-05_bs4_ga4_r16_a32_d0p05_20260616_104036_001 ===

=== Grid run 2/18: tennis_from_tiser_e1_lr5e-05_bs4_ga8_r16_a32_d0p05_20260616_104036_002 ===

=== Grid run 3/18: SKIP existing combo ===
e=1|lr=0.0001|bs=4|ga=4|r=16|alpha=32|dropout=0.05
existing run: tennis_from_tiser_e1_lr0.0001_bs4_ga4_r16_a32_d0p05_20260616_095331_002
adapter: model/tennis_from_tiser_e1_lr0.0001_bs4_ga4_r16_a32_d0p05_20260616_095331_002/adapter

=== Grid run 4/18: SKIP existing combo ===
e=1|lr=0.0001|bs=4|ga=8|r=16|alpha=32|dropout=0.05
existing run: tennis_from_tiser_e1_lr0.0001_bs4_ga8_r16_a32_d0p05_20260616_095331_004
adapter: model/tennis_from_tiser_e1_lr0.0001_bs4_ga8_r16_a32_d0p05_20260616_095331_004/adapter

=== Grid run 5/18: SKIP existing combo ===
e=1|lr=0.0002|bs=4|ga=4|r=16|alpha=32|dropout=0.05
existing run: tennis_from_tiser_e1_lr0.0002_bs4_ga4_r16_a32_d0p05_20260616_095331_006
adapter: model/tennis_from_tiser_e1_lr0.0002_bs4_ga4_r16_a32_d0p05_20260616_0

## 10. Aggregation

In [ ]:
run_shell(f'''
python scripts/tennis/compare_adapters.py \
  --results-dir "{RESULTS_DIR}"
''', enabled=RUN_AGGREGATION)

if Path('scripts/tennis/aggregate_tennis_results.py').exists():
    run_shell(f'''
python scripts/tennis/aggregate_tennis_results.py \
  --results-dir "{RESULTS_DIR}"
''', enabled=RUN_AGGREGATION)
else:
    print('scripts/tennis/aggregate_tennis_results.py not found; skipping optional aggregation.')

In [13]:
# Evaluate the base tiser adapter on the tennis dataset to establish a baseline score to be compared to adapters fine-tuned on tennis dataset
from pathlib import Path
import subprocess
import sys

ORIGINAL_TISER_7B_CONDITION = "original_tiser_qwen7b_test224"
ORIGINAL_TISER_7B_OUTPUT_DIR = (
    Path("results/tennis_from_tiser_experiments/scored")
    / ORIGINAL_TISER_7B_CONDITION
)

command = [
    sys.executable,
    "scripts/tennis/evaluate_tennis.py",
    "--config", CONFIG,
    "--test-file", TENNIS_TEST,
    "--adapter-dir", "checkpoints/adapter",
    "--condition", ORIGINAL_TISER_7B_CONDITION,
    "--batch-size", "8",
    "--max-new-tokens", "256",
    "--output-dir", ORIGINAL_TISER_7B_OUTPUT_DIR.as_posix(),
]

print(" ".join(command))
subprocess.run(command, check=True)

print("Wrote:")
print(ORIGINAL_TISER_7B_OUTPUT_DIR / "metrics.json")
print(ORIGINAL_TISER_7B_OUTPUT_DIR / "metrics_report.md")
print(ORIGINAL_TISER_7B_OUTPUT_DIR / "predictions.jsonl")

/usr/bin/python3 scripts/tennis/evaluate_tennis.py --config config/config_tennis.yaml --test-file data/tennis/tennis_test.json --adapter-dir checkpoints/adapter --condition original_tiser_qwen7b_test224 --batch-size 8 --max-new-tokens 256 --output-dir results/tennis_from_tiser_experiments/scored/original_tiser_qwen7b_test224
Wrote:
results/tennis_from_tiser_experiments/scored/original_tiser_qwen7b_test224/metrics.json
results/tennis_from_tiser_experiments/scored/original_tiser_qwen7b_test224/metrics_report.md
results/tennis_from_tiser_experiments/scored/original_tiser_qwen7b_test224/predictions.jsonl


In [13]:
from pathlib import Path
import json
import subprocess
import sys

RESULTS_DIR = "results/tennis_domain_adaptation"
SCORED_DIR = Path(RESULTS_DIR) / "scored"

EVAL_BATCH_SIZE = 8          # safe; use 4 or 8 on A100 if you want faster eval
EVAL_MAX_NEW_TOKENS = 256
EVAL_LIMIT = None            # None = full tennis test set; set 5 for smoke

def adapter_complete(adapter_dir):
    adapter_dir = Path(adapter_dir)
    return (
        adapter_dir.is_dir()
        and (adapter_dir / "adapter_config.json").exists()
        and (adapter_dir / "adapter_model.safetensors").exists()
    )

def safe_condition_name(name):
    return (
        str(name)
        .replace("/", "__")
        .replace(" ", "_")
        .replace(":", "-")
    )

# Collect completed grid adapters.
adapters = {}

for manifest_path in sorted(Path("gridsearch").glob("tennis_from_tiser_qwen7b_*/manifest.json")):
    rows = json.loads(manifest_path.read_text())
    for row in rows:
        if row.get("status") not in {"ok", "skipped_existing"}:
            continue

        adapter_dir = row.get("adapter_dir")
        run_name = row.get("run_name")
        combo_key = row.get("combo_key")

        if not adapter_dir or not run_name:
            continue
        if not adapter_complete(adapter_dir):
            print(f"[skip missing adapter] {run_name}: {adapter_dir}")
            continue

        # One eval per hyperparameter combo.
        adapters[combo_key or run_name] = {
            "run_name": run_name,
            "adapter_dir": adapter_dir,
            "combo_key": combo_key,
        }

print(f"Found {len(adapters)} completed adapters to evaluate.")

eval_manifest = []

for index, item in enumerate(adapters.values(), start=1):
    run_name = item["run_name"]
    adapter_dir = item["adapter_dir"]
    condition = safe_condition_name(run_name)
    output_dir = SCORED_DIR / condition
    metrics_path = output_dir / "metrics.json"

    if metrics_path.exists():
        print(f"\n=== Eval {index}/{len(adapters)}: SKIP existing ===")
        print(metrics_path)
        eval_manifest.append({
            **item,
            "condition": condition,
            "output_dir": output_dir.as_posix(),
            "metrics_path": metrics_path.as_posix(),
            "status": "skipped_existing",
        })
        continue

    command = [
        sys.executable,
        "scripts/tennis/evaluate_tennis.py",
        "--config", CONFIG,
        "--test-file", TENNIS_TEST,
        "--adapter-dir", adapter_dir,
        "--condition", condition,
        "--batch-size", str(EVAL_BATCH_SIZE),
        "--max-new-tokens", str(EVAL_MAX_NEW_TOKENS),
        "--output-dir", output_dir.as_posix(),
    ]

    if EVAL_LIMIT is not None:
        command.extend(["--limit", str(EVAL_LIMIT)])

    print(f"\n=== Eval {index}/{len(adapters)}: {condition} ===")
    print(" ".join(command))

    result = subprocess.run(command)
    status = "ok" if result.returncode == 0 else "failed"

    eval_manifest.append({
        **item,
        "condition": condition,
        "output_dir": output_dir.as_posix(),
        "metrics_path": metrics_path.as_posix(),
        "status": status,
    })

    Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
    (Path(RESULTS_DIR) / "grid_eval_manifest.json").write_text(
        json.dumps(eval_manifest, indent=2)
    )

    if result.returncode != 0:
        raise RuntimeError(f"Evaluation failed: {condition}")

print("\nAll evaluations complete.")
print(f"Wrote {Path(RESULTS_DIR) / 'grid_eval_manifest.json'}")

# Aggregate/compare all scored runs.
subprocess.run([
    sys.executable,
    "scripts/tennis/compare_adapters.py",
    "--results-dir", RESULTS_DIR,
], check=True)

print("Comparison files:")
print(Path(RESULTS_DIR) / "comparisons/adapter_comparison.md")
print(Path(RESULTS_DIR) / "comparisons/adapter_comparison.csv")
print(Path(RESULTS_DIR) / "comparisons/adapter_comparison.json")

Found 25 completed adapters to evaluate.

=== Eval 1/25: SKIP existing ===
results/tennis_domain_adaptation/scored/tennis_from_tiser_e1_lr0.0001_bs4_ga4_r16_a32_d0p0_20260616_095331_001/metrics.json

=== Eval 2/25: SKIP existing ===
results/tennis_domain_adaptation/scored/tennis_from_tiser_e1_lr0.0001_bs4_ga4_r16_a32_d0p05_20260616_095331_002/metrics.json

=== Eval 3/25: SKIP existing ===
results/tennis_domain_adaptation/scored/tennis_from_tiser_e1_lr0.0001_bs4_ga8_r16_a32_d0p0_20260616_095331_003/metrics.json

=== Eval 4/25: SKIP existing ===
results/tennis_domain_adaptation/scored/tennis_from_tiser_e1_lr0.0001_bs4_ga8_r16_a32_d0p05_20260616_095331_004/metrics.json

=== Eval 5/25: SKIP existing ===
results/tennis_domain_adaptation/scored/tennis_from_tiser_e1_lr0.0002_bs4_ga4_r16_a32_d0p0_20260616_095331_005/metrics.json

=== Eval 6/25: tennis_from_tiser_e1_lr0.0002_bs4_ga4_r16_a32_d0p05_20260616_095331_006 ===
/usr/bin/python3 scripts/tennis/evaluate_tennis.py --config config/config_t

## 11. Display result summaries

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

summary_files = [
    Path(RESULTS_DIR) / 'comparisons/adapter_comparison.md',
    Path(RESULTS_DIR) / 'comparisons/final_results_table.md',
    Path(RESULTS_DIR) / 'comparisons/category_analysis.md',
    Path(RESULTS_DIR) / 'comparisons/forgetting_analysis.md',
]

for path in summary_files:
    if path.exists():
        display(Markdown(f'### {path.as_posix()}'))
        display(Markdown(path.read_text(encoding='utf-8')))
    else:
        print(f'[missing] {path.as_posix()}')

## 12. Save artifacts

In [ ]:
from pathlib import Path
import shutil

COPY_ZIP_TO_DRIVE = False
DRIVE_ZIP_TARGET = Path('/content/drive/MyDrive/tennis_domain_adaptation_results.zip')

results_path = Path(RESULTS_DIR)
if not results_path.exists():
    print(f'Results directory does not exist yet: {results_path}')
else:
    archive_path = shutil.make_archive(
        'tennis_domain_adaptation_results',
        'zip',
        root_dir=results_path.parent,
        base_dir=results_path.name,
    )
    print(f'Wrote {archive_path}')
    if COPY_ZIP_TO_DRIVE:
        DRIVE_ZIP_TARGET.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(archive_path, DRIVE_ZIP_TARGET)
        print(f'Copied to {DRIVE_ZIP_TARGET}')

## 13. Safety notes

- Do not run full 7B evaluations unless GPU memory is sufficient.
- Start with `--limit 5`.
- The smoke config using Qwen2.5-0.5B is only for base-model smoke checks, not for 7B adapters.
- Adapter model must match Qwen/Qwen2.5-7B-Instruct.
- Training should only be run after `tennis_train_traced.json` has validated TISER outputs.
- Keep checkpoints and large generated outputs out of git; final experiment summaries should live under `results/tennis_domain_adaptation/`.